# AF2FS1 — shared P3 frequency selection
AF2 ditambah spatially-varying low/high-frequency selection pada shared P3.

In [ ]:
ARM='AF2FS1'
exec('''from google.colab import drive
drive.mount('/content/drive')
import importlib,json,os,shutil,subprocess,sys,tarfile,time,torch
from pathlib import Path
assert torch.cuda.is_available(),'Aktifkan T4 GPU.'
REPO=Path('/content/coffee-bean-detection'); BRANCH='codex/af2-complementary-mechanisms'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    result=subprocess.run(clone)
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
else: raise RuntimeError('Git clone gagal tiga kali.')
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
from coffee_detector.drive_project import resolve_drive_project_root,require_project_artifact
REQ=('bundles/faruq-development-v3-grouped.tar','experiments/faruq-v3-breadth-screening-batch-v1/candidates/AFAB/AF2_seed42/weights/best.pt','experiments/faruq-v3-af2-complement-v1/static_audit.json')
PROJECT=resolve_drive_project_root(required_relative_paths=REQ)
ARCHIVE=require_project_artifact(PROJECT,REQ[0]); AF2=require_project_artifact(PROJECT,REQ[1]); STATIC=require_project_artifact(PROJECT,REQ[2])
DATA=Path('/content/faruq-development-v3-grouped')
if not (DATA/'data.yaml').is_file():
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
assert (DATA/'data.yaml').is_file() and not (DATA/'test').exists()
OUTPUT=PROJECT/'experiments/faruq-v3-af2-complement-v1'
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_complement_arm','--arm',ARM,'--data-root',str(DATA),'--grouped-summary',str(DATA/'faruq_grouped_summary.json'),'--af2-checkpoint',str(AF2),'--static-audit',str(STATIC),'--output-root',str(OUTPUT),'--seed','42','--device','0','--authorize-training']
log=OUTPUT/f'{ARM}_seed42_run.log'; log.parent.mkdir(parents=True,exist_ok=True)
print('START/RESUME:',ARM,'| log=',log,flush=True)
with log.open('a',encoding='utf-8') as handle:
    process=subprocess.Popen(command,cwd=REPO,stdout=handle,stderr=subprocess.STDOUT)
    while process.poll() is None:
        csv=OUTPUT/ARM/f'{ARM}_seed42/results.csv'
        epochs=max(0,len(csv.read_text(errors='replace').splitlines())-1) if csv.is_file() else 0
        print(f'{ARM}: {epochs}/30 epoch tercatat',flush=True); time.sleep(300)
if process.returncode:
    print('\n'.join(log.read_text(errors='replace').splitlines()[-150:])); raise RuntimeError(f'{ARM} gagal: {process.returncode}')
result=json.loads((OUTPUT/'val_reports'/f'{ARM}_seed42_result.json').read_text())
print(json.dumps(result,indent=2,ensure_ascii=False))
''')
